# Управление циклом, генераторы списков и функции

В этом ноутбуке мы продолжаем изучать Python на уровне, который нужен начинающему аналитику данных.

На прошлом этапе мы уже познакомились со строками, списками, словарями и циклами. Теперь научимся делать код более управляемым и повторно используемым.

**Цели занятия:**

1. понять, как Python хранит значения и почему переменные могут ссылаться на один объект;
2. научиться управлять циклом с помощью `break` и `continue`;
3. познакомиться с генераторами списков, или `list comprehension`;
4. научиться создавать собственные функции через `def`;
5. разобраться, чем `print()` отличается от `return`;
6. собрать маленькие функции, полезные для аналитических расчетов.

## 1. Короткое повторение: переменная и значение

Переменная — это имя, по которому мы обращаемся к значению.

Например, если мы пишем `my_var = 30`, то создаем имя `my_var`, связанное со значением `30`.

Для начинающего уровня достаточно помнить:

- переменная хранит не саму «коробку», а ссылку на значение;
- несколько переменных могут ссылаться на одно и то же значение;
- если мы изменяем переменную, она может начать ссылаться на другое значение.

In [1]:
my_var = 30
print(my_var)

30


Функция `id()` показывает внутренний идентификатор объекта. Это не нужно использовать в обычной аналитике каждый день, но пример помогает понять, что переменные могут ссылаться на один и тот же объект.

In [2]:
print(id(my_var))

10755696


Создадим еще две переменные, которые получают то же значение.

In [3]:
a = my_var
b = my_var

print("my_var =", my_var, "id:", id(my_var))
print("a      =", a, "id:", id(a))
print("b      =", b, "id:", id(b))

my_var = 30 id: 10755696
a      = 30 id: 10755696
b      = 30 id: 10755696


Теперь изменим только переменную `b`. Значения `my_var` и `a` останутся прежними.

Это хороший момент для обсуждения: в Python мы часто не «переписываем значение внутри коробки», а связываем имя переменной с новым объектом.

In [4]:
b += 1

print("my_var =", my_var, "id:", id(my_var))
print("a      =", a, "id:", id(a))
print("b      =", b, "id:", id(b))

my_var = 30 id: 10755696
a      = 30 id: 10755696
b      = 31 id: 10755728


### Контрольный вопрос

Почему после команды `b += 1` значения `my_var` и `a` не изменились?

## 2. Управление циклом: `break`

Цикл нужен, когда одну и ту же операцию нужно повторить несколько раз.

Иногда цикл нужно остановить раньше, чем он закончится сам. Для этого используется команда `break`.

Простая логика:

> «Иди по данным, пока не найдешь нужное. Как только нашел — остановись».

В аналитике это может быть поиск первого заказа с ошибкой, первого клиента с нужным статусом или первой даты, где показатель превысил план.

In [5]:
# Пример: ищем первую продажу больше 100000 рублей
sales = [35000, 58000, 72000, 125000, 99000, 150000]

for sale in sales:
    print("Проверяем продажу:", sale)
    if sale > 100000:
        print("Найдена первая крупная продажа:", sale)
        break

Проверяем продажу: 35000
Проверяем продажу: 58000
Проверяем продажу: 72000
Проверяем продажу: 125000
Найдена первая крупная продажа: 125000


Обратите внимание: после нахождения продажи `125000` цикл остановился. Значения после нее уже не проверялись.

### Пример с `while True`

Конструкция `while True` означает «повторять бесконечно». Чтобы такой цикл не стал действительно бесконечным, внутри обязательно должно быть условие остановки с `break`.

In [6]:
i = 0

while True:
    print("Текущий шаг:", i)
    if i > 2:
        print("Условие остановки выполнено")
        break
    i += 1

Текущий шаг: 0
Текущий шаг: 1
Текущий шаг: 2
Текущий шаг: 3
Условие остановки выполнено


## 3. Управление циклом: `continue`

Команда `continue` не останавливает весь цикл. Она пропускает только текущий шаг и переходит к следующему.

Простая логика:

> «Если значение нам не подходит, не обрабатывай его, переходи к следующему».

В аналитике это полезно, когда в данных есть неподходящие записи: нулевые продажи, пустые значения, тестовые клиенты, отмененные заказы.

In [7]:
# Пример: считаем только положительные продажи
sales = [12000, 0, 45000, -5000, 30000, 0, 78000]

valid_sales = []

for sale in sales:
    if sale <= 0:
        print("Пропускаем некорректное значение:", sale)
        continue
    valid_sales.append(sale)

print("Корректные продажи:", valid_sales)

Пропускаем некорректное значение: 0
Пропускаем некорректное значение: -5000
Пропускаем некорректное значение: 0
Корректные продажи: [12000, 45000, 30000, 78000]


### `break` и `continue`: разница

| Команда | Что делает | Пример логики |
|---|---|---|
| `break` | полностью завершает цикл | «нашли нужное — остановились» |
| `continue` | пропускает текущий шаг | «это значение не подходит — идем дальше» |

## 4. Практика: минимальный делитель числа

Задача из исходного файла:

> Программа получает натуральное число `n > 1`. Нужно вывести минимальный делитель этого числа, отличный от единицы.

Например, для числа `12` делителями являются `1, 2, 3, 4, 6, 12`. Минимальный делитель, отличный от 1, — это `2`.

Для учебного ноутбука мы не будем использовать `input()`, чтобы файл можно было запускать целиком. Значение числа зададим в переменной.

In [8]:
n = 12

d = 2
while True:
    if n % d == 0:
        print("Минимальный делитель числа", n, "равен", d)
        break
    d += 1

Минимальный делитель числа 12 равен 2


### Пошаговый разбор

1. `n = 12` — число, которое проверяем.
2. `d = 2` — начинаем проверять делители с 2, потому что 1 нам не подходит.
3. `n % d == 0` — проверяем, делится ли число `n` на `d` без остатка.
4. Если делится — печатаем ответ и останавливаем цикл через `break`.
5. Если не делится — увеличиваем `d` на 1.

### Практика для слушателя

Измените значение `n` и проверьте минимальный делитель для чисел `15`, `17`, `21`, `49`.

In [9]:
for n in [15, 17, 21, 49]:
    d = 2
    while True:
        if n % d == 0:
            print(f"Минимальный делитель числа {n}: {d}")
            break
        d += 1

Минимальный делитель числа 15: 3
Минимальный делитель числа 17: 17
Минимальный делитель числа 21: 3
Минимальный делитель числа 49: 7


## 5. Практика: пропуск чисел и досрочная остановка

Задача из исходного файла:

> Перебрать все числа от `a` до `b` включительно.  
> Не выводить числа, которые делятся на 2 или на 3.  
> Если встретилось число, кратное 777, нужно принудительно закончить цикл.

В этой задаче одновременно используются `continue` и `break`.

In [10]:
a = 1
b = 20

current = a
while current <= b:
    if current % 777 == 0:
        print("Нашли число, кратное 777. Останавливаем цикл:", current)
        break

    if current % 2 == 0 or current % 3 == 0:
        current += 1
        continue

    print(current)
    current += 1

1
5
7
11
13
17
19


### Что здесь происходит

- если число кратно `777`, используется `break`, потому что цикл нужно завершить полностью;
- если число делится на `2` или `3`, используется `continue`, потому что нужно пропустить только это число;
- если число прошло обе проверки, оно выводится на экран.

## 6. Генераторы списков: `list comprehension`

Генератор списка, или `list comprehension`, — это короткий способ создать новый список.

Сначала посмотрим обычный способ через цикл.

In [11]:
squares = []

for number in range(1, 6):
    squares.append(number ** 2)

print(squares)

[1, 4, 9, 16, 25]


Теперь запишем то же самое короче.

In [12]:
squares = [number ** 2 for number in range(1, 6)]
print(squares)

[1, 4, 9, 16, 25]


Общая форма:

```python
[выражение for элемент in коллекция]
```

Читаем справа налево:

1. берем элементы из коллекции;
2. применяем к каждому элементу выражение;
3. собираем результаты в новый список.

In [13]:
# Пример 1: числа от 0 до 9
lst1 = [i for i in range(10)]

# Пример 2: символы строки
lst2 = [symbol for symbol in "hello"]

# Пример 3: расчет по формуле
lst3 = [x**2 - 3*x + 7 for x in range(10)]

print("lst1:", lst1)
print("lst2:", lst2)
print("lst3:", lst3)

lst1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
lst2: ['h', 'e', 'l', 'l', 'o']
lst3: [7, 5, 5, 7, 11, 17, 25, 35, 47, 61]


## 7. Генератор списка с условием

Можно добавить условие `if`, чтобы в новый список попали только подходящие элементы.

In [14]:
sales = [12000, 0, 45000, -5000, 30000, 0, 78000]

positive_sales = [sale for sale in sales if sale > 0]
print(positive_sales)

[12000, 45000, 30000, 78000]


Такой прием часто используется в аналитике для простой фильтрации данных.

Например:

- оставить только положительные продажи;
- выбрать только клиентов из нужного города;
- оставить только заказы со статусом `paid`;
- убрать пустые или ошибочные значения.

In [15]:
orders = [
    {"id": 1, "status": "paid", "amount": 12000},
    {"id": 2, "status": "cancelled", "amount": 0},
    {"id": 3, "status": "paid", "amount": 45000},
    {"id": 4, "status": "new", "amount": 15000},
]

paid_amounts = [order["amount"] for order in orders if order["status"] == "paid"]
print(paid_amounts)

[12000, 45000]


### Практика: список из 100 нулей

Задача из исходного файла:

> При помощи генератора списка сохраните в переменной `zeroes` список из 100 нулей.

In [16]:
zeroes = [0 for i in range(100)]

print("Количество элементов:", len(zeroes))
print("Первые 10 элементов:", zeroes[:10])

Количество элементов: 100
Первые 10 элементов: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## 8. Практика: список делителей числа

Задача из исходного файла:

> Создать список, состоящий из делителей числа.

Для числа `10` результат будет `[1, 2, 5, 10]`.

In [17]:
n = 10

divisors = [i for i in range(1, n + 1) if n % i == 0]
print(divisors)

[1, 2, 5, 10]


### Разбор строки

```python
divisors = [i for i in range(1, n + 1) if n % i == 0]
```

- `range(1, n + 1)` перебирает числа от 1 до `n` включительно;
- `if n % i == 0` оставляет только те числа, на которые `n` делится без остатка;
- `i` в начале означает, что в новый список мы кладем сам найденный делитель.

## 9. Функции: зачем они нужны

Функция — это именованный блок кода, который можно вызвать много раз.

Функции нужны, чтобы:

- не копировать один и тот же код;
- давать понятные имена действиям;
- разбивать большую задачу на маленькие шаги;
- повторно использовать расчеты.

В аналитике функции удобно использовать для расчета KPI, проверки качества данных, подготовки признаков, форматирования отчетов.

Общий шаблон функции:

```python
def имя_функции(параметры):
    тело_функции
```

Важные детали:

1. определение функции начинается с `def`;
2. после имени функции идут круглые скобки;
3. после скобок ставится двоеточие;
4. тело функции пишется с отступом;
5. функция выполняется только тогда, когда мы ее вызываем.

In [18]:
def say_hello():
    print("Здравствуйте!")
    print("Сегодня мы продолжаем изучать Python.")

Выше мы только создали функцию. Теперь ее нужно вызвать.

In [19]:
say_hello()

Здравствуйте!
Сегодня мы продолжаем изучать Python.


## 10. Функция с параметром

Параметр — это значение, которое мы передаем в функцию.

Например, функция ниже принимает число `x` и печатает его квадрат.

In [20]:
def calculate_square(x):
    print(f"Квадрат числа {x} = {x ** 2}")

In [21]:
calculate_square(5)
calculate_square(3)
calculate_square(2)

Квадрат числа 5 = 25
Квадрат числа 3 = 9
Квадрат числа 2 = 4


Одна функция — три разных вызова. Это и есть повторное использование кода.

### Практика: функция `hello_world`

Создадим функцию `hello_world`, которая выводит сообщение `Hello, world!`.

In [22]:
def hello_world():
    print("Hello, world!")

hello_world()

Hello, world!


## 11. Функция для суммы чисел от 1 до `t`

Задача из исходного файла:

> Написать функцию `summa_n`, которая принимает целое положительное число `t` и находит сумму всех чисел от 1 до `t` включительно.

In [23]:
def summa_n(t):
    total = 0
    for number in range(1, t + 1):
        total += number
    print(f"Я знаю, что сумма чисел от 1 до {t} равна {total}")

summa_n(5)

Я знаю, что сумма чисел от 1 до 5 равна 15


### Почему `range(1, t + 1)`?

Функция `range()` не включает правую границу.

Поэтому, если нам нужны числа от `1` до `5` включительно, нужно написать:

```python
range(1, 6)
```

Именно поэтому для переменной `t` используется `t + 1`.

## 12. Функция для суммы цифр в строке

Задача из исходного файла:

> Написать функцию `sum_num`, которая принимает строку, находит в ней все цифры и суммирует их.

Например, строка `123QwertY321` содержит цифры `1, 2, 3, 3, 2, 1`. Их сумма равна `12`.

In [24]:
def sum_num(text):
    total = 0
    for symbol in text:
        if symbol.isdigit():
            total += int(symbol)
    print(total)

sum_num("123QwertY321")

12


### Где это может пригодиться аналитику

Например, в данных встречаются текстовые коды заказов: `order-2026-015`. Иногда нужно извлечь или проверить числовые части строки.

В реальных задачах для сложного извлечения обычно используют регулярные выражения, но на базовом уровне важно понять сам принцип перебора символов.

In [25]:
order_code = "order-2026-015"
sum_num(order_code)

16


## 13. Функция проверки учебного кода доступа

Задача из исходного файла:

> Написать функцию `check_access_code`, которая проверяет учебный код доступа на сложность.

Условия сложного кода доступа:

1. есть хотя бы 3 цифры;
2. есть хотя бы одна заглавная буква;
3. есть хотя бы один символ из набора `!@#$%*`;
4. общая длина не менее 10 символов.

Если все условия выполнены, функция выводит `Код подходит`, иначе — `Код нужно усилить`.

In [26]:
def check_access_code(access_code):
    digits = 0
    uppercase_letters = 0
    special_symbols = 0
    allowed_special_symbols = "!@#$%*"

    for symbol in access_code:
        if symbol.isdigit():
            digits += 1
        if symbol.isupper():
            uppercase_letters += 1
        if symbol in allowed_special_symbols:
            special_symbols += 1

    length = len(access_code)

    if digits >= 3 and uppercase_letters >= 1 and special_symbols >= 1 and length >= 10:
        print("Код подходит")
    else:
        print("Код нужно усилить")

In [27]:
check_access_code("Code1357)")
check_access_code("Code1357!")

Easy peasy
Perfect password


### Разбор логики

В функции мы считаем не сам код доступа, а его признаки:

- сколько в нем цифр;
- сколько заглавных букв;
- сколько специальных символов;
- какая у него длина.

После этого проверяем все условия вместе через оператор `and`.

## 14. `print()` и `return`: важное различие

Очень частая ошибка начинающих — путать `print()` и `return`.

| Инструмент | Что делает | Где видим результат | Можно ли использовать результат дальше? |
|---|---|---|---|
| `print()` | выводит значение на экран | в выводе ячейки | нет, если дополнительно не сохранить |
| `return` | возвращает значение из функции | внутри программы | да |

Если функция только печатает результат, другой код не может напрямую использовать этот результат в расчетах.

In [28]:
def print_five():
    print(5)

result = print_five()
print("Что сохранилось в result:", result)

5
Что сохранилось в result: None


Функция напечатала `5`, но в переменную `result` попало значение `None`. Это означает «ничего не возвращено».

In [29]:
def return_five():
    return 5

result = return_five()
print("Что сохранилось в result:", result)
print("Теперь можно использовать в расчете:", result + 1)

Что сохранилось в result: 5
Теперь можно использовать в расчете: 6


## 15. Функция, которая возвращает результат

Перепишем расчет квадрата так, чтобы функция не печатала результат, а возвращала его.

In [30]:
def get_square(x):
    return x ** 2

square_5 = get_square(5)
print(square_5)
print(square_5 + 100)

25
125


Такой подход чаще используется в аналитическом коде: функция считает значение, возвращает его, а дальше мы сами решаем, что с ним делать.

## 16. Практика: функция `is_person_teenager`

Задача из исходного файла:

> Считать человека подростком, если его возраст находится в пределах от 12 до 17 лет.  
> Функция `is_person_teenager` принимает возраст и возвращает `True` или `False`.

In [31]:
def is_person_teenager(age):
    return age >= 12 and age <= 17

print(is_person_teenager(10))
print(is_person_teenager(12))
print(is_person_teenager(15))
print(is_person_teenager(18))

False
True
True
False


Более компактная запись:

In [32]:
def is_person_teenager(age):
    return 12 <= age <= 17

for age in [10, 12, 15, 18]:
    print(age, "->", is_person_teenager(age))

10 -> False
12 -> True
15 -> True
18 -> False


## 17. Мини-практика для аналитика: функции для заказов

Теперь соберем маленький аналитический пример.

У нас есть список заказов. Каждый заказ — это словарь с номером, статусом и суммой.

Нужно:

1. оставить только оплаченные заказы;
2. посчитать общую выручку;
3. найти первый крупный заказ;
4. рассчитать средний чек.

In [33]:
orders = [
    {"id": 101, "status": "paid", "amount": 12000},
    {"id": 102, "status": "cancelled", "amount": 0},
    {"id": 103, "status": "paid", "amount": 45000},
    {"id": 104, "status": "new", "amount": 15000},
    {"id": 105, "status": "paid", "amount": 78000},
    {"id": 106, "status": "paid", "amount": 150000},
]

In [34]:
def get_paid_orders(order_list):
    return [order for order in order_list if order["status"] == "paid"]

paid_orders = get_paid_orders(orders)
print(paid_orders)

[{'id': 101, 'status': 'paid', 'amount': 12000}, {'id': 103, 'status': 'paid', 'amount': 45000}, {'id': 105, 'status': 'paid', 'amount': 78000}, {'id': 106, 'status': 'paid', 'amount': 150000}]


In [35]:
def calculate_revenue(order_list):
    revenue = 0
    for order in order_list:
        revenue += order["amount"]
    return revenue

revenue = calculate_revenue(paid_orders)
print("Выручка по оплаченным заказам:", revenue)

Выручка по оплаченным заказам: 285000


In [36]:
def find_first_large_order(order_list, threshold):
    for order in order_list:
        if order["amount"] >= threshold:
            return order
    return None

large_order = find_first_large_order(paid_orders, 100000)
print("Первый крупный заказ:", large_order)

Первый крупный заказ: {'id': 106, 'status': 'paid', 'amount': 150000}


In [37]:
def calculate_average_check(order_list):
    if len(order_list) == 0:
        return 0
    return calculate_revenue(order_list) / len(order_list)

average_check = calculate_average_check(paid_orders)
print("Средний чек:", average_check)

Средний чек: 71250.0


### Что мы закрепили в мини-практике

- `list comprehension` помог отфильтровать оплаченные заказы;
- функция `calculate_revenue()` вернула числовой результат;
- функция `find_first_large_order()` использовала ранний выход через `return`;
- функция `calculate_average_check()` использовала проверку, чтобы избежать деления на ноль.

## 18. Типичные ошибки начинающих

### Ошибка 1. Забыли вызвать функцию

```python
def hello():
    print("Привет")
```

Этот код только создает функцию. Чтобы она выполнилась, нужно написать:

```python
hello()
```

### Ошибка 2. Перепутали `print()` и `return`

Если результат нужен для дальнейших расчетов, используйте `return`.

### Ошибка 3. Забыли увеличить счетчик в `while`

Если счетчик не меняется, цикл может стать бесконечным.

### Ошибка 4. Слишком сложный `list comprehension`

Если строка стала трудной для чтения, лучше написать обычный цикл. Читаемость важнее краткости.

## Итоги занятия

Сегодня мы разобрали:

- как переменные могут ссылаться на значения;
- как использовать `break` для полной остановки цикла;
- как использовать `continue` для пропуска одного шага цикла;
- как создавать списки через `list comprehension`;
- как объявлять и вызывать функции;
- чем `print()` отличается от `return`;
- как применять функции в маленьких аналитических задачах.

Следующий логичный шаг — перейти к работе с табличными данными через `pandas`: `DataFrame`, столбцы, строки, группировки и визуализации.